# 15 — Skills and the AgentSkills Plugin

## Overview

As agents grow more capable, stuffing every instruction into one giant system prompt
becomes unwieldy. Strands solves this with **skills**: reusable instruction bundles stored
as `SKILL.md` files that the agent loads on demand.

The `AgentSkills` plugin manages the lifecycle:
1. It scans a directory (or a list) for skills at startup.
2. It injects a compact `<available_skills>` summary into the system prompt.
3. When the agent decides a skill is relevant, it calls the built-in `skills` tool to load
   the full `SKILL.md` instructions — keeping the context window lean for unrelated turns.

**What you will build:**
- A customer-support agent with mock tools for returns, product info, and troubleshooting
- A `skills/` directory with two skill packages, each with a `SKILL.md` and reference files
- An `AgentSkills` plugin wired into the agent
- A programmatically created skill added at runtime without touching the filesystem

## Prerequisites

- Python 3.10 or later
- AWS account with [Amazon Bedrock](https://aws.amazon.com/bedrock/) access configured
- Basic familiarity with the Strands `Agent` class —
  see [01-first-agent](../01-first-agent/) if needed

Run this notebook from the `15-skills/` directory so that relative paths to `./skills/`
resolve correctly.

In [ ]:
%pip install -U strands-agents strands-agents-tools -q

## Step 1: Define the agent tools

Skills tell the agent *how* to answer; tools give it *the means* to answer.
Here we define three mock tools that simulate a real customer-support backend:

| Tool | Purpose |
|---|---|
| `get_return_policy` | Look up return window, conditions, and refund timeline by product category |
| `get_product_info` | Retrieve specs, warranty, and compatibility information |
| `get_technical_support` | Fetch step-by-step troubleshooting guides by symptom |

In [ ]:
from strands import tool


@tool
def get_return_policy(product_category: str) -> str:
    """Return policy for a product category (e.g. 'laptops', 'smartphones', 'accessories')."""
    policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
    }
    default = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection",
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies",
    }
    p = policies.get(product_category.lower(), default)
    return (
        f"Return Policy \u2014 {product_category.title()}:\n"
        f"  Return window : {p['window']} from delivery\n"
        f"  Condition     : {p['condition']}\n"
        f"  Process       : {p['process']}\n"
        f"  Refund time   : {p['refund_time']}\n"
        f"  Shipping      : {p['shipping']}\n"
        f"  Warranty      : {p['warranty']}"
    )


@tool
def get_product_info(product_type: str) -> str:
    """Technical specifications and warranty details for a product type."""
    catalog = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32 GB RAM, SSD storage, various display sizes",
            "features": "Backlit keyboards, USB-C/Thunderbolt, Wi-Fi 6, Bluetooth 5.0",
            "compatibility": "Windows 11, macOS, Linux (varies by model)",
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G/4G, 128 GB\u20131 TB storage, multi-lens cameras",
            "features": "Wireless charging, water resistance, biometric security",
            "compatibility": "iOS/Android, carrier-unlocked options available",
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless, active noise cancellation, 20 Hz\u201320 kHz",
            "features": "Touch controls, voice assistant, USB-C charging",
            "compatibility": "Bluetooth 5.0+, 3.5 mm jack",
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p, IPS/OLED panels, various sizes",
            "features": "HDR, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
        },
    }
    product = catalog.get(product_type.lower())
    if not product:
        return f"Specs for '{product_type}' not found. Contact technical support."
    return (
        f"Product Info \u2014 {product_type.title()}:\n"
        f"  Warranty       : {product['warranty']}\n"
        f"  Specifications : {product['specs']}\n"
        f"  Key Features   : {product['features']}\n"
        f"  Compatibility  : {product['compatibility']}"
    )


@tool
def get_technical_support(issue_description: str) -> str:
    """Retrieve troubleshooting steps for a described hardware or connectivity issue."""
    guides = {
        "power": (
            "Device won't power on \u2014 checklist:\n"
            "1. Verify the power cable is firmly connected at both ends.\n"
            "2. Try a different wall outlet.\n"
            "3. Hold the power button for 10 seconds to force a hard reset.\n"
            "4. If the battery is removable, remove it, wait 30 s, reinsert, and retry.\n"
            "5. Check for any visible damage (swollen battery, burnt smell).\n"
            "If none of these resolve the issue, escalate to hardware support."
        ),
        "wifi": (
            "Wi-Fi / connectivity issue \u2014 checklist:\n"
            "1. Toggle Wi-Fi off and back on.\n"
            "2. Forget the network and reconnect.\n"
            "3. Restart the router and the device.\n"
            "4. Check for firmware/driver updates.\n"
            "5. Test on a different network to isolate the problem."
        ),
        "setup": (
            "Initial setup guidance:\n"
            "1. Charge the device to at least 50 % before first use.\n"
            "2. Follow the on-screen setup wizard.\n"
            "3. Connect to Wi-Fi and install available updates before proceeding.\n"
            "4. Register the device to activate the warranty."
        ),
        "overheat": (
            "Overheating \u2014 checklist:\n"
            "1. Ensure vents are not blocked; use on a hard, flat surface.\n"
            "2. Close unused applications to reduce CPU/GPU load.\n"
            "3. Check for malware or runaway background processes.\n"
            "4. Clean dust from vents with compressed air if accessible.\n"
            "5. If the device shuts down due to heat, stop use and contact support."
        ),
    }
    issue_lower = issue_description.lower()
    for keyword, guide in guides.items():
        if keyword in issue_lower:
            return guide
    return (
        f"No specific guide found for: '{issue_description}'.\n"
        "General advice: restart the device, check for software updates, "
        "and consult the manufacturer's support portal if the issue persists."
    )


print("Tools defined.")

## Step 2: Inspect the skill packages

Each skill lives in its own subdirectory under `skills/`. The minimum requirement is a
`SKILL.md` file with a YAML front-matter block containing `name` and `description`.
The body is the full instruction set the agent receives when it activates the skill.

```
skills/
├── returns-policy/
│   ├── SKILL.md                   ← front-matter + instructions
│   └── references/
│       └── returns-checklist.md   ← referenced by the skill via file_read
└── technical-troubleshooting/
    ├── SKILL.md
    └── references/
        └── escalation-guide.md
```

The `allowed-tools` front-matter field is advisory: it tells the agent which tools are
relevant when this skill is active.

In [ ]:
from pathlib import Path

for path in sorted(Path('./skills').glob('*/SKILL.md')):
    print(f"{'=' * 60}")
    print(f"Skill: {path.parent.name}")
    print('=' * 60)
    print(path.read_text())

## Step 3: Create the AgentSkills plugin

`AgentSkills` accepts:
- a **path to a single skill directory** — loads that one skill
- a **path to a parent directory** — loads every child that contains a `SKILL.md`
- a **list** mixing paths and programmatic `Skill(...)` objects

The plugin reads each skill's `name` and `description` from the front-matter and injects
them into the system prompt as a compact XML block. The full `SKILL.md` body is only
fetched when the agent activates the skill.

In [ ]:
from strands import AgentSkills

plugin = AgentSkills(skills='./skills/')

print('Available skills:')
for skill in plugin.get_available_skills():
    print(f'  {skill.name}: {skill.description}')

## Step 4: Build the agent

Three things come together here:

1. **`plugins=[plugin]`** — wires the `AgentSkills` plugin in so the agent gets the
   `<available_skills>` block and the `skills` activation tool.
2. **`tools=[...]`** — the real domain tools the agent calls to answer questions.
   Skills declare which tools they need in `allowed-tools`; those tools must appear here.
3. **`file_read`** from `strands-agents-tools` — lets activated skills open the
   `references/` files declared in their `SKILL.md`.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools import file_read

model = BedrockModel(
    model_id="us.amazon.nova-lite-v1:0",
    temperature=0.3,
)

SYSTEM_PROMPT = """You are a helpful customer support assistant for an electronics retailer.
Use your tools to answer questions accurately.
Activate a skill when the user's request clearly matches one of the available domains."""

agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    plugins=[plugin],
    tools=[
        get_return_policy,
        get_product_info,
        get_technical_support,
        file_read,
    ],
)

print('Agent ready.')

## Step 5: Ask questions that should trigger different skills

The first question is about returns — it should activate `returns-policy`.
The second is a troubleshooting question — it should activate `technical-troubleshooting`.

In [ ]:
response_returns = agent(
    'I opened my laptop box and now I think I need to return it. '
    'What is the return policy and what does the warranty cover?'
)

In [ ]:
response_tech = agent(
    'My laptop will not power on after charging overnight. '
    'What troubleshooting steps should I try first?'
)

## Step 6: Check which skills were activated

`plugin.get_activated_skills(agent)` returns the list of skill names the agent loaded
during the session. This is useful for logging, analytics, or routing decisions.

In [ ]:
activated = plugin.get_activated_skills(agent)
print('Skills activated this session:', activated)

## Step 7: Add a skill programmatically at runtime

You do not need a file on disk to create a skill. The `Skill` dataclass lets you define
one in Python — useful for per-session instructions, environment-specific playbooks, or
dynamically assembled workflows.

`set_available_skills` **replaces** the current list, so pass the existing skills plus
the new one to extend rather than replace.

In [ ]:
from strands import Skill

handoff_skill = Skill(
    name="conversation-summary",
    description=(
        "Summarize a support interaction into a short handoff note. "
        "Use when the user asks for a summary, handoff, or recap of the conversation."
    ),
    instructions=(
        "When the user asks for a summary or handoff note, produce a concise bullet "
        "summary with: issue described, steps already tried, and recommended next action."
    ),
)

plugin.set_available_skills(plugin.get_available_skills() + [handoff_skill])

print('Updated skill list:')
for skill in plugin.get_available_skills():
    print(f'  {skill.name}')

In [ ]:
response_summary = agent(
    'Can you give me a quick handoff summary of our conversation so far?'
)

## What you learned

| Concept | Key point |
|---|---|
| `SKILL.md` structure | Front-matter (`name`, `description`, `allowed-tools`) + instruction body |
| `AgentSkills` plugin | Scans a directory; injects a compact skill summary; loads full instructions on demand |
| Skill activation | Agent calls the built-in `skills` tool; full `SKILL.md` body enters context only then |
| Reference files | Skills can call `file_read` on files under `references/` for additional guidance |
| Programmatic skills | `Skill(name, description, instructions)` creates a skill without touching the filesystem |
| Runtime updates | `plugin.set_available_skills(...)` replaces the list; takes effect on the next agent call |

### Next steps

- Add a `scripts/` directory to a skill and use the `shell` tool to run environment-specific checks
- Combine skills with [06-memory](../06-memory/) for personalized, session-aware workflows
- Use skills to separate language-specific instructions in multilingual support agents